# EDA Tuned End-to-End Comparison

Notebook này dùng để so sánh công bằng giữa baseline model và pipeline sau khai phá dữ liệu:

- Load dữ liệu clean chung.
- Split train/validation/test trước khi fit tham số feature engineering.
- Tạo EDA-based features và interaction features.
- Preprocess numeric/categorical columns.
- Train Random Forest/SVM variants.
- Tune threshold theo validation F1-score.
- Chọn model tốt nhất trên validation và đánh giá test set một lần cuối.


## 1. Import và cấu hình

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "eda_tuned_end_to_end_comparison.py").exists():
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "train" / "feature_engineering" / "eda_tuned_end_to_end_comparison.py").exists():
    NOTEBOOK_DIR = CURRENT_DIR / "train" / "feature_engineering"
else:
    raise FileNotFoundError("Cannot find eda_tuned_end_to_end_comparison.py")

sys.path.insert(0, str(NOTEBOOK_DIR))

import eda_tuned_end_to_end_comparison as pipeline

OUTPUT_DIR = pipeline.OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook dir:", NOTEBOOK_DIR)
print("Data path:", pipeline.DATA_PATH)
print("Output dir:", OUTPUT_DIR)

Notebook dir: D:\HocTap\KT&XLTT\CUOIKI\train\feature_engineering
Data path: D:\HocTap\KT&XLTT\CUOIKI\data\diabetic_data_clean_common.csv
Output dir: D:\HocTap\KT&XLTT\CUOIKI\train\feature_engineering\eda_tuned_comparison_outputs


## 2. Load dữ liệu và split train/validation/test

Chia dữ liệu trước khi fit các ngưỡng như Q3 để tránh leakage từ validation/test.

In [2]:
df = pd.read_csv(pipeline.DATA_PATH)

y = df[pipeline.TARGET_COLUMN]
X = df.drop(columns=[pipeline.TARGET_COLUMN])

from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=pipeline.RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=pipeline.RANDOM_STATE,
    stratify=y_train_val,
)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(X_train), len(X_val), len(X_test)],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
})

display(split_summary)

,split,rows,positive_rate
0,train,65128,0.460892
1,validation,16282,0.460877
2,test,20353,0.460915


## 3. Feature engineering từ EDA

Các ngưỡng học được như Q3 chỉ fit trên train, sau đó áp dụng cho validation/test.

In [3]:
fe_params = pipeline.fit_feature_engineering_params(X_train)

X_train_fe = pipeline.add_eda_features(X_train, fe_params)
X_val_fe = pipeline.add_eda_features(X_val, fe_params)
X_test_fe = pipeline.add_eda_features(X_test, fe_params)

X_train_model = X_train_fe.drop(columns=[col for col in pipeline.DROP_COLUMNS if col in X_train_fe.columns])
X_val_model = X_val_fe.drop(columns=[col for col in pipeline.DROP_COLUMNS if col in X_val_fe.columns])
X_test_model = X_test_fe.drop(columns=[col for col in pipeline.DROP_COLUMNS if col in X_test_fe.columns])

new_features = [
    "total_prior_visits",
    "has_prior_inpatient",
    "meds_per_day",
    "labs_per_day",
    "is_long_stay",
    "is_senior",
    "a1c_abnormal",
    "insulin_changed",
    "num_diabetes_drugs_used",
    "num_unique_diag_groups",
    "prior_inpatient_long_stay",
    "insulin_change_a1c_abnormal",
    "high_medication_load",
    "high_lab_load",
]

pd.DataFrame([fe_params]).to_csv(OUTPUT_DIR / "feature_engineering_params.csv", index=False)
pd.DataFrame({"new_feature": new_features}).to_csv(OUTPUT_DIR / "new_features.csv", index=False)

print("Feature engineering params:")
display(pd.DataFrame([fe_params]))
display(pd.DataFrame({"new_feature": new_features}))

Feature engineering params:


,long_stay_q3,high_med_q3,high_lab_q3
0,6.0,20.0,57.0


,new_feature
0,total_prior_visits
1,has_prior_inpatient
2,meds_per_day
3,labs_per_day
4,is_long_stay
5,is_senior
6,a1c_abnormal
7,insulin_changed
8,num_diabetes_drugs_used
9,num_unique_diag_groups


## 4. Preprocess dữ liệu

Numeric columns dùng `StandardScaler`, categorical columns dùng `OneHotEncoder`.

In [4]:
preprocessor = pipeline.build_preprocessor(X_train_model)

X_train_processed = preprocessor.fit_transform(X_train_model)
X_val_processed = preprocessor.transform(X_val_model)
X_test_processed = preprocessor.transform(X_test_model)

feature_names = pipeline.get_feature_names(preprocessor)
pd.DataFrame({"feature_name": feature_names}).to_csv(OUTPUT_DIR / "encoded_feature_names.csv", index=False)

print("Train processed shape:", X_train_processed.shape)
print("Validation processed shape:", X_val_processed.shape)
print("Test processed shape:", X_test_processed.shape)
display(pd.DataFrame({"feature_name": feature_names}).head(20))

Train processed shape: (65128, 267)
Validation processed shape: (16282, 267)
Test processed shape: (20353, 267)


,feature_name
0,num__admission_type_id
1,num__discharge_disposition_id
2,num__admission_source_id
3,num__time_in_hospital
4,num__num_lab_procedures
5,num__num_procedures
6,num__num_medications
7,num__number_outpatient
8,num__number_emergency
9,num__number_inpatient


## 5. Train model và tune threshold trên validation

Mỗi model được đánh giá bằng 2 cách:

- `default`: ngưỡng mặc định, ví dụ Random Forest dùng 0.5.
- `tuned_for_validation_f1`: chọn threshold tốt nhất theo F1-score trên validation.

In [5]:
comparison_rows = []
comparison_rows = pipeline.add_baseline_rows(comparison_rows)

trained_models = {}
best_validation = None

for model_name, model in pipeline.make_models().items():
    print("Training:", model_name)
    import time
    start = time.time()
    model.fit(X_train_processed, y_train)
    train_time_sec = time.time() - start
    trained_models[model_name] = model

    val_scores = pipeline.score_values(model, X_val_processed)
    default_t = pipeline.default_threshold(model)
    tuned_t, tuned_f1 = pipeline.tune_threshold(y_val, val_scores)

    for strategy, threshold in [("default", default_t), ("tuned_for_validation_f1", tuned_t)]:
        y_val_pred = pipeline.predict_with_threshold(val_scores, threshold)
        metrics = pipeline.evaluate_predictions(y_val, y_val_pred, val_scores)
        row = {
            "experiment": "eda_features_tuned_pipeline",
            "model": model_name,
            "threshold_strategy": strategy,
            "threshold": threshold,
            **metrics,
            "train_time_sec": train_time_sec,
        }
        comparison_rows.append(row)

        safe_name = model_name.lower().replace(" ", "_").replace("+", "plus")
        pipeline.save_confusion_matrix(
            y_val,
            y_val_pred,
            OUTPUT_DIR / f"{safe_name}_{strategy}_validation_confusion_matrix.csv",
        )

        pd.DataFrame(
            pipeline.classification_report(y_val, y_val_pred, output_dict=True, zero_division=0)
        ).T.to_csv(OUTPUT_DIR / f"{safe_name}_{strategy}_validation_classification_report.csv")

        if best_validation is None or row["f1_score"] > best_validation["f1_score"]:
            best_validation = row.copy()

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ["f1_score", "roc_auc", "accuracy"], ascending=False
)
comparison_df.insert(0, "rank", range(1, len(comparison_df) + 1))
comparison_df.to_csv(OUTPUT_DIR / "baseline_vs_eda_tuned_validation_comparison.csv", index=False)

display(comparison_df)

Training: Random Forest + EDA
Training: Random Forest + EDA balanced
Training: Random Forest + EDA conservative
Training: SVM + EDA


,rank,experiment,model,threshold_strategy,threshold,accuracy,precision,recall,f1_score,roc_auc,train_time_sec
7,1,eda_features_tuned_pipeline,Random Forest + EDA conservative,tuned_for_validation_f1,0.400161,0.599374,0.542136,0.841018,0.659284,0.696456,7.467508
5,2,eda_features_tuned_pipeline,Random Forest + EDA balanced,tuned_for_validation_f1,0.400198,0.607788,0.549760,0.823028,0.659195,0.696596,7.872087
3,3,eda_features_tuned_pipeline,Random Forest + EDA,tuned_for_validation_f1,0.355000,0.583712,0.529560,0.866604,0.657400,0.691378,5.562762
9,4,eda_features_tuned_pipeline,SVM + EDA,tuned_for_validation_f1,-0.195617,0.576833,0.525481,0.843683,0.647606,0.671280,12.085538
0,5,baseline_from_model_comparison,Random Forest,default,NaN,0.644823,0.614810,0.614072,0.614441,0.699756,NaN
6,6,eda_features_tuned_pipeline,Random Forest + EDA conservative,default,0.500000,0.643963,0.621529,0.581690,0.600950,0.696456,7.467508
4,7,eda_features_tuned_pipeline,Random Forest + EDA balanced,default,0.500000,0.642182,0.622589,0.567830,0.593950,0.696596,7.872087
1,8,baseline_from_model_comparison,SVM,default,NaN,0.623941,0.594940,0.576626,0.585640,0.668293,NaN
8,9,eda_features_tuned_pipeline,SVM + EDA,default,0.000000,0.626213,0.598912,0.572095,0.585196,0.671280,12.085538
2,10,eda_features_tuned_pipeline,Random Forest + EDA,default,0.500000,0.640953,0.631420,0.530784,0.576745,0.691378,5.562762


## 6. Đánh giá test set cho model tốt nhất

Test set chỉ dùng một lần sau khi đã chọn model bằng validation F1-score.

In [6]:
best_model_name = best_validation["model"]
best_threshold = best_validation["threshold"]
best_model = trained_models[best_model_name]

test_scores = pipeline.score_values(best_model, X_test_processed)
y_test_pred = pipeline.predict_with_threshold(test_scores, best_threshold)
test_metrics = pipeline.evaluate_predictions(y_test, y_test_pred, test_scores)

test_result = {
    "selected_by": "best_validation_f1",
    "model": best_model_name,
    "threshold_strategy": best_validation["threshold_strategy"],
    "threshold": best_threshold,
    **test_metrics,
}

pd.DataFrame([test_result]).to_csv(OUTPUT_DIR / "final_test_metrics.csv", index=False)
pipeline.save_confusion_matrix(y_test, y_test_pred, OUTPUT_DIR / "final_test_confusion_matrix.csv")
pd.DataFrame(
    pipeline.classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
).T.to_csv(OUTPUT_DIR / "final_test_classification_report.csv")

joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")
joblib.dump(best_model, OUTPUT_DIR / "best_validation_model.joblib")

display(pd.DataFrame([test_result]))

,selected_by,model,threshold_strategy,threshold,accuracy,precision,recall,f1_score,roc_auc
0,best_validation_f1,Random Forest + EDA conservative,tuned_for_validation_f1,0.400161,0.598192,0.541526,0.836158,0.657337,0.698067


## 7. Output chính để đưa vào báo cáo

- `baseline_vs_eda_tuned_validation_comparison.csv`: bảng so sánh validation chính.
- `final_test_metrics.csv`: kết quả test set của model được chọn theo validation.
- `new_features.csv`: danh sách feature mới từ EDA.
- `encoded_feature_names.csv`: danh sách feature sau preprocessing.
